# RTMScore 虚拟筛选教程

**论文引用:** Shen C, Zhang X, Deng Y, et al. *Boosting Protein-Ligand Binding Pose Prediction and Virtual Screening Based on Residue-Atom Distance Likelihood Potential and Graph Transformer.* Journal of Medicinal Chemistry, 2022, 65(15): 10691-10706.

## 核心创新

RTMScore 与传统打分函数的关键区别在于：它**不直接预测结合亲和力**，而是使用**混合密度网络 (Mixture Density Network, MDN)** 来学习蛋白质残基与配体原子之间的**距离分布**。

核心原理：
- **训练阶段**：模型学习用高斯混合分布来拟合残基-原子对之间的真实距离
- **预测阶段**：将观测到的距离代入学到的分布，计算其似然 (likelihood)
- 似然越高，说明该复合物的几何构型越符合模型学到的"正常"结合模式
- 最终，似然之和作为打分值 (score)，与结合亲和力相关联

这种间接方法的优势：
1. 不需要直接回归亲和力值，避免了标签稀疏的问题
2. 距离信息是丰富且容易获取的监督信号
3. 学到的分布具有可解释性 -- 每个高斯分量可以对应不同的相互作用类型

## 学习目标

通过本教程，你将学到：
1. 如何提取蛋白质残基和配体原子的特征
2. 如何构建残基-原子距离矩阵
3. 混合密度网络 (MDN) 的原理和实现
4. 如何利用距离似然进行蛋白-配体打分
5. 评估打分函数性能的方法（Pearson R、RMSE）

> **注意：** 本教程是简化版教学模型，真实 RTMScore 使用图变换器 (Graph Transformer) 提取更深层的残基和原子表示，性能远优于此简化模型。

In [ ]:
# ---- 项目根目录定位与路径设置 ----

def find_project_root(marker='demo_data'):
    """向上查找包含 demo_data/ 的项目根目录"""
    from pathlib import Path
    here = Path('.').resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f'无法找到包含 {marker}/ 的项目根目录')

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'demo_data'
CORESET_FILE = DATA_DIR / 'CoreSet.dat'
COMPLEX_DIR = DATA_DIR / 'coreset'

print(f'项目根目录: {PROJECT_ROOT}')
print(f'数据目录:   {DATA_DIR}')

# ---- 导入依赖库 ----

import os
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.distributions import Normal
from rdkit import Chem, RDLogger
from Bio.PDB.PDBParser import PDBParser
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

%matplotlib inline

# 抑制 RDKit 和 BioPython 的警告信息
RDLogger.logger().setLevel(RDLogger.ERROR)
warnings.filterwarnings('ignore')

# ---- 版本信息 ----
import rdkit
import Bio
import scipy
import sklearn

version_info = pd.DataFrame({
    '库': ['PyTorch', 'RDKit', 'BioPython', 'NumPy', 'SciPy', 'scikit-learn', 'pandas'],
    '版本': [
        torch.__version__,
        rdkit.__version__,
        Bio.__version__,
        np.__version__,
        scipy.__version__,
        sklearn.__version__,
        pd.__version__
    ]
})
display(version_info)

## 1. 超参数设置

RTMScore 教学模型的关键超参数说明：

| 超参数 | 含义 |
|--------|------|
| `N_GAUSSIANS` | 高斯混合分量的数量，越多越能拟合复杂的多峰距离分布 |
| `RESIDUE_FEAT_DIM` | 残基特征维度（20种标准氨基酸 + 1个"其他"类别） |
| `ATOM_FEAT_DIM` | 配体原子特征维度（9种元素 one-hot + 是否芳香性） |
| `DIST_THRESHOLD` | 距离阈值，仅考虑距离小于此值的残基-原子对 |
| `HIDDEN_DIM` | 隐藏层维度 |
| `BATCH_SIZE` | 设为 1，因为每个复合物大小不同 |

In [ ]:
# ---- 超参数定义 ----

N_GAUSSIANS = 10           # 高斯混合分量的数量 -- 越多越能拟合复杂的多峰距离分布
RESIDUE_FEAT_DIM = 21      # 20种标准氨基酸 + 1个"其他"类别
ATOM_FEAT_DIM = 10         # 9种元素类型 one-hot + 是否芳香性
DIST_THRESHOLD = 7.0       # 仅考虑距离小于此阈值的残基-原子对 (单位: 埃)
HIDDEN_DIM = 128           # 隐藏层维度
N_EPOCHS = 200             # 训练轮数
LR = 1e-3                  # 学习率
BATCH_SIZE = 1             # 由于每个复合物大小不同, 使用 batch_size=1
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'计算设备: {DEVICE}')
print(f'高斯分量数: {N_GAUSSIANS}')
print(f'距离阈值: {DIST_THRESHOLD} A')
print(f'隐藏层维度: {HIDDEN_DIM}')
print(f'训练轮数: {N_EPOCHS}')

## 2. 数据加载与特征提取

RTMScore 需要提取两类特征：

1. **残基特征**：使用 BioPython 解析口袋 PDB 文件
   - 21 维 one-hot 编码（20 种标准氨基酸 + 1 个 "其他" 类别）
   - CA 原子坐标作为残基位置
   - 收集每个残基所有原子的坐标（用于计算最小距离）

2. **配体原子特征**：使用 RDKit 解析 mol2/sdf 文件
   - 10 维特征向量（9 种元素 one-hot + 是否芳香性）
   - 3D 坐标

3. **距离矩阵**：计算每对残基-配体原子之间的最小距离
   - 使用残基内所有原子（而非仅 CA）到配体原子的最小距离
   - 侧链原子才是直接参与相互作用的部分

In [ ]:
def parse_coreset(path):
    """解析 CoreSet.dat 文件, 返回 {pdbid: logKa} 字典

    文件格式: 第一行以 '#' 开头的表头, 后续行以空白分隔
    列0=PDB代码, 列3=logKa (结合亲和力的对数值)
    """
    labels = {}
    with open(path, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            parts = line.strip().split()
            if len(parts) >= 4:
                pdbid = parts[0]
                logKa = float(parts[3])
                labels[pdbid] = logKa
    return labels


# 20种标准氨基酸列表
STANDARD_AAS = [
    'ALA', 'ARG', 'ASN', 'ASP', 'CYS',
    'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
    'LEU', 'LYS', 'MET', 'PHE', 'PRO',
    'SER', 'THR', 'TRP', 'TYR', 'VAL'
]


def residue_features(resname):
    """将残基名转为 21 维 one-hot 向量

    前20维对应20种标准氨基酸, 第21维表示"其他"(非标准氨基酸)
    """
    feat = np.zeros(RESIDUE_FEAT_DIM, dtype=np.float32)
    if resname in STANDARD_AAS:
        feat[STANDARD_AAS.index(resname)] = 1.0
    else:
        feat[-1] = 1.0  # OTHER 类别
    return feat


def atom_features(atom):
    """将 RDKit 原子转为 10 维特征向量

    前9维: 元素类型 one-hot (C, N, O, S, F, Cl, Br, I, P)
    第10维: 是否为芳香性原子
    """
    ELEMENTS = ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'I', 'P']
    feat = np.zeros(ATOM_FEAT_DIM, dtype=np.float32)
    symbol = atom.GetSymbol()
    if symbol in ELEMENTS:
        feat[ELEMENTS.index(symbol)] = 1.0
    feat[9] = 1.0 if atom.GetIsAromatic() else 0.0
    return feat


def parse_pocket_residues(pocket_pdb):
    """解析口袋 PDB 文件, 提取残基级别的特征和坐标

    对每个标准残基:
      - 计算残基特征 (21维 one-hot)
      - 提取 CA 原子坐标作为残基位置
      - 收集该残基所有原子的坐标 (用于计算最小距离)

    返回:
      res_feats: (N_r, 21) -- 残基特征矩阵
      ca_positions: (N_r, 3) -- CA 原子坐标
      res_atoms_coords: list of arrays, 每个元素形状 (n_atoms, 3) -- 残基内所有原子坐标
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('pocket', pocket_pdb)

    res_feats_list = []
    ca_positions_list = []
    res_atoms_coords_list = []

    for model in structure.get_list():
        for chain in model:
            for residue in chain:
                # 只保留标准氨基酸残基 (跳过水分子和 HETATM)
                # residue.get_id()[0] == ' ' 表示标准残基
                if residue.get_id()[0] != ' ':
                    continue

                resname = residue.get_resname()

                # 检查是否有 CA 原子, 没有则跳过
                if 'CA' not in residue:
                    continue

                # 残基特征
                feat = residue_features(resname)
                res_feats_list.append(feat)

                # CA 原子坐标
                ca_coord = residue['CA'].get_vector().get_array()
                ca_positions_list.append(ca_coord)

                # 收集该残基所有原子的坐标
                atom_coords = []
                for atom in residue.get_atoms():
                    atom_coords.append(atom.get_vector().get_array())
                res_atoms_coords_list.append(np.array(atom_coords, dtype=np.float32))

    if len(res_feats_list) == 0:
        return None, None, None

    res_feats = np.array(res_feats_list, dtype=np.float32)
    ca_positions = np.array(ca_positions_list, dtype=np.float32)

    return res_feats, ca_positions, res_atoms_coords_list


def compute_min_distance_matrix(res_atoms_coords, lig_positions):
    """计算残基-配体原子之间的最小距离矩阵

    对于每对 (残基i, 配体原子j), 计算残基i中任意原子与配体原子j之间的最小距离。
    使用最小距离而非 CA 距离, 因为侧链原子才是直接参与相互作用的部分。

    参数:
      res_atoms_coords: list of (n_atoms_in_res, 3) arrays -- 每个残基的所有原子坐标
      lig_positions: (N_l, 3) -- 配体原子坐标

    返回:
      dist_matrix: (N_r, N_l) -- 最小距离矩阵
    """
    N_r = len(res_atoms_coords)
    N_l = lig_positions.shape[0]
    dist_matrix = np.zeros((N_r, N_l), dtype=np.float32)

    for i in range(N_r):
        # res_atoms_coords[i]: (n_atoms, 3)
        # lig_positions: (N_l, 3)
        # 计算残基i的每个原子与每个配体原子的距离
        # diff: (n_atoms, N_l, 3)
        diff = res_atoms_coords[i][:, np.newaxis, :] - lig_positions[np.newaxis, :, :]
        # dists: (n_atoms, N_l)
        dists = np.sqrt((diff ** 2).sum(axis=-1))
        # 取每个配体原子到残基i中最近原子的距离
        dist_matrix[i] = dists.min(axis=0)

    return dist_matrix


def load_complex_rtm(pdbid, labels):
    """加载一个蛋白-配体复合物的全部数据

    步骤:
    1. 用 BioPython 解析口袋 PDB -> 残基特征和原子坐标
    2. 用 RDKit 加载配体 mol2 文件 -> 原子特征和坐标
    3. 计算最小距离矩阵
    4. 生成距离阈值掩码 (只关注近距离的残基-原子对)

    返回 dict 或 None (失败时)
    """
    pocket_pdb = str(COMPLEX_DIR / pdbid / f"{pdbid}_pocket.pdb")
    ligand_mol2 = str(COMPLEX_DIR / pdbid / f"{pdbid}_ligand.mol2")
    ligand_sdf = str(COMPLEX_DIR / pdbid / f"{pdbid}_ligand.sdf")

    # 解析口袋残基
    if not os.path.exists(pocket_pdb):
        return None
    res_feats, _, res_atoms_coords = parse_pocket_residues(pocket_pdb)
    if res_feats is None or len(res_feats) == 0:
        return None

    # 加载配体 (优先 mol2, 失败则尝试 sdf)
    mol = Chem.MolFromMol2File(ligand_mol2, sanitize=False)
    if mol is None and os.path.exists(ligand_sdf):
        mol = Chem.MolFromMolFile(ligand_sdf, sanitize=False)
    if mol is None:
        return None

    try:
        Chem.SanitizeMol(mol)
    except Exception:
        pass  # 即使 sanitize 失败也继续, 仍然可以提取特征

    # 提取配体原子特征和坐标
    conf = mol.GetConformer()
    lig_feats_list = []
    lig_positions_list = []
    for atom in mol.GetAtoms():
        lig_feats_list.append(atom_features(atom))
        pos = conf.GetAtomPosition(atom.GetIdx())
        lig_positions_list.append([pos.x, pos.y, pos.z])

    if len(lig_feats_list) == 0:
        return None

    lig_feats = np.array(lig_feats_list, dtype=np.float32)
    lig_positions = np.array(lig_positions_list, dtype=np.float32)

    # 计算最小距离矩阵: (N_r, N_l)
    dist_matrix = compute_min_distance_matrix(res_atoms_coords, lig_positions)

    # 掩码: 只保留距离小于阈值的残基-原子对
    # 这些近距离对才是有意义的相互作用
    mask = dist_matrix < DIST_THRESHOLD

    # 需要至少有 2 个有效对 (太少会影响学习)
    if mask.sum() < 2:
        return None

    logKa = labels.get(pdbid, None)
    if logKa is None:
        return None

    return {
        'res_feats': res_feats,
        'lig_feats': lig_feats,
        'dist_matrix': dist_matrix,
        'mask': mask,
        'label': logKa
    }

In [ ]:
# ---- 加载数据并展示一个样本 ----

labels = parse_coreset(CORESET_FILE)
print(f'CoreSet 中共有 {len(labels)} 个复合物')

# 展示前几个复合物的标签
sample_labels = list(labels.items())[:5]
display(pd.DataFrame(sample_labels, columns=['PDB ID', 'logKa']))

# 加载一个示例复合物并展示其数据
example_pdbid = sample_labels[0][0]
example_data = load_complex_rtm(example_pdbid, labels)

if example_data is not None:
    print(f'\n示例复合物: {example_pdbid}')
    print(f'  残基特征形状:   {example_data["res_feats"].shape}  (N_r, 21)')
    print(f'  配体特征形状:   {example_data["lig_feats"].shape}  (N_l, 10)')
    print(f'  距离矩阵形状:   {example_data["dist_matrix"].shape}  (N_r, N_l)')
    print(f'  有效残基-原子对: {example_data["mask"].sum()} / {example_data["mask"].size}')
    print(f'  logKa:          {example_data["label"]:.2f}')
else:
    print(f'示例复合物 {example_pdbid} 加载失败')

## 3. 数据集与数据加载器

RTMScore 数据集的特殊之处：

- **训练目标是距离分布**（MDN loss），而**不是 logKa 值**
- logKa 仅用于最终评估，不参与训练
- 由于每个复合物的残基/原子数不同，使用 `batch_size=1`

每个样本包含：
- `res_feats`: 残基特征 `(N_r, 21)` 
- `lig_feats`: 配体原子特征 `(N_l, 10)`
- `dist_matrix`: 残基-原子距离矩阵 `(N_r, N_l)` -- MDN 的训练目标
- `mask`: 距离阈值掩码 `(N_r, N_l)`
- `logKa`: 结合亲和力对数值 -- 仅用于评估

In [ ]:
class RTMDataset(Dataset):
    """RTMScore 数据集

    每个样本包含:
      - res_feats: 残基特征 (N_r, 21)
      - lig_feats: 配体原子特征 (N_l, 10)
      - dist_matrix: 残基-原子距离矩阵 (N_r, N_l) -- 这是 MDN 的训练目标
      - mask: 距离阈值掩码 (N_r, N_l)
      - logKa: 结合亲和力对数值 -- 仅用于评估, 不参与训练

    注意: 训练目标是距离分布 (MDN loss), 而不是 logKa 值!
    """

    def __init__(self, data_list):
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        d = self.data[idx]
        return (
            torch.FloatTensor(d['res_feats']),
            torch.FloatTensor(d['lig_feats']),
            torch.FloatTensor(d['dist_matrix']),
            torch.BoolTensor(d['mask']),
            torch.FloatTensor([d['label']])
        )


# ---- 加载全部数据并划分训练/测试集 ----

all_data = []
failed = 0
for pdbid in sorted(labels.keys()):
    result = load_complex_rtm(pdbid, labels)
    if result is not None:
        result['pdbid'] = pdbid
        all_data.append(result)
    else:
        failed += 1

print(f'成功加载: {len(all_data)} 个复合物')
print(f'加载失败: {failed} 个复合物')

# 划分训练集/测试集 (80/20)
np.random.shuffle(all_data)
split = int(0.8 * len(all_data))
train_data = all_data[:split]
test_data = all_data[split:]

train_dataset = RTMDataset(train_data)
test_dataset = RTMDataset(test_data)

# BATCH_SIZE=1 因为每个复合物的残基/原子数不同, 无法简单 batch
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 展示数据集统计
split_info = pd.DataFrame({
    '数据集': ['训练集', '测试集', '合计'],
    '样本数': [len(train_data), len(test_data), len(all_data)]
})
display(split_info)

# 展示数据形状示例
sample = train_dataset[0]
shape_info = pd.DataFrame({
    '张量': ['res_feats', 'lig_feats', 'dist_matrix', 'mask', 'label'],
    '形状': [str(tuple(s.shape)) for s in sample],
    '数据类型': [str(s.dtype) for s in sample]
})
display(shape_info)

## 4. 模型架构

RTMScore 的核心是**混合密度网络 (Mixture Density Network, MDN)**。

### 模型结构

1. **残基嵌入层**：将 21 维残基特征映射到隐藏空间
2. **配体原子嵌入层**：将 10 维原子特征映射到隐藏空间
3. **配对 MLP**：将残基-原子对的拼接特征变换为隐藏表示
4. **MDN 头部**：输出高斯混合分布的参数

### MDN 核心思想

MDN 输出三个参数来定义一个混合高斯分布：

| 参数 | 含义 | 激活函数 | 约束 |
|------|------|----------|------|
| $\pi$ (pi) | 混合系数 | Softmax | 各分量权重和为 1 |
| $\mu$ (mu) | 均值 | ELU + 1 | 确保正值（距离为正） |
| $\sigma$ (sigma) | 标准差 | ELU + 1.1 | 确保正值且有下界 |

$$p(d | \text{residue}, \text{atom}) = \sum_{k=1}^{K} \pi_k \cdot \mathcal{N}(d | \mu_k, \sigma_k)$$

其中 $d$ 是残基-原子距离，$K$ 是高斯分量数。

In [ ]:
class RTMScoreToyModel(nn.Module):
    """RTMScore 简化教学模型

    模型结构:
    1. 残基嵌入层: 将 21 维残基特征映射到隐藏空间
    2. 配体原子嵌入层: 将 10 维原子特征映射到隐藏空间
    3. 配对 MLP: 将残基-原子对的拼接特征变换为隐藏表示
    4. MDN 头部: 输出高斯混合分布的参数 (pi, mu, sigma)

    MDN 核心思想:
    - pi: 混合系数 (每个高斯分量的权重, 通过 softmax 归一化)
    - mu: 均值 (每个高斯分量的中心位置, 通过 ELU+1 确保正值)
    - sigma: 标准差 (每个高斯分量的宽度, 通过 ELU+1.1 确保正值且有下界)
    """

    def __init__(self, res_dim=RESIDUE_FEAT_DIM, atom_dim=ATOM_FEAT_DIM,
                 hidden_dim=HIDDEN_DIM, n_gaussians=N_GAUSSIANS):
        super().__init__()
        self.res_embed = nn.Linear(res_dim, hidden_dim)
        self.lig_embed = nn.Linear(atom_dim, hidden_dim)

        # 配对 MLP -- 处理残基-原子对的拼接特征
        # 使用 LayerNorm 替代 BatchNorm1d, 避免 batch_size=1 时的问题
        self.pair_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ELU(),
            nn.Dropout(0.15)
        )

        # MDN 输出头
        self.pi_net = nn.Linear(hidden_dim, n_gaussians)    # 混合系数
        self.mu_net = nn.Linear(hidden_dim, n_gaussians)    # 均值
        self.sigma_net = nn.Linear(hidden_dim, n_gaussians) # 标准差

    def forward(self, res_feats, lig_feats, mask):
        """前向传播

        参数:
          res_feats: (N_r, res_dim) -- 残基特征
          lig_feats: (N_l, atom_dim) -- 配体原子特征
          mask: (N_r, N_l) -- 布尔掩码, 标记有效的残基-原子对

        返回:
          pi: (K, n_gaussians) -- 混合系数
          mu: (K, n_gaussians) -- 均值
          sigma: (K, n_gaussians) -- 标准差
          其中 K = mask.sum() 为有效对的数量
        """
        # 嵌入
        res_h = self.res_embed(res_feats)   # (N_r, hidden_dim)
        lig_h = self.lig_embed(lig_feats)   # (N_l, hidden_dim)

        N_r, N_l = res_h.size(0), lig_h.size(0)

        # 构建所有残基-原子对的特征
        # 通过广播机制展开为 (N_r, N_l, hidden_dim) 的张量
        res_exp = res_h.unsqueeze(1).expand(-1, N_l, -1)   # (N_r, N_l, hidden_dim)
        lig_exp = lig_h.unsqueeze(0).expand(N_r, -1, -1)   # (N_r, N_l, hidden_dim)
        pair_h = torch.cat([res_exp, lig_exp], dim=-1)      # (N_r, N_l, hidden_dim*2)

        # 使用掩码筛选有效对, 展平为 (K, hidden_dim*2)
        h = self.pair_mlp(pair_h[mask])  # (K, hidden_dim)

        # MDN 头部输出高斯混合分布的三个参数
        pi = F.softmax(self.pi_net(h), dim=-1)     # 混合权重, softmax 确保和为 1
        mu = F.elu(self.mu_net(h)) + 1             # 均值, ELU+1 确保正值
        sigma = F.elu(self.sigma_net(h)) + 1.1     # 标准差, ELU+1.1 确保正值且有下界

        return pi, mu, sigma


def mdn_loss(pi, sigma, mu, y, eps=1e-10):
    """MDN 损失函数: 高斯混合分布的负对数似然

    原理: 给定距离 y, 计算它在混合高斯分布下的概率, 取负对数作为损失

    数学公式:
      p(y) = sum_k pi_k * N(y | mu_k, sigma_k)
      loss = -log p(y) = -log sum_k pi_k * N(y | mu_k, sigma_k)

    使用 logsumexp 技巧避免数值下溢:
      loss = -logsumexp(log(pi_k) + log N(y | mu_k, sigma_k))

    参数:
      pi: (K, n_gaussians) -- 混合系数
      sigma: (K, n_gaussians) -- 标准差
      mu: (K, n_gaussians) -- 均值
      y: (K,) -- 真实距离值

    返回:
      标量损失值 (所有有效对的平均负对数似然)
    """
    normal = Normal(mu, sigma)
    # y.unsqueeze(-1): (K, 1) -> 广播为 (K, n_gaussians)
    loglik = normal.log_prob(y.unsqueeze(-1).expand_as(mu))
    # log-sum-exp 计算混合分布的对数概率
    loss = -torch.logsumexp(torch.log(pi + eps) + loglik, dim=-1)
    return loss.mean()


def compute_score(model, res_feats, lig_feats, dist_matrix, mask):
    """计算 RTMScore 打分值

    核心思想: 打分 = 所有有效残基-原子对的距离概率之和

    直觉解释:
    - 模型学到了"好的结合"中残基-原子距离应该遵循的分布
    - 对于一个新的复合物, 将其真实距离代入学到的分布
    - 如果该复合物是一个好的结合者, 其距离应该高度符合学到的分布
    - 因此概率之和 (score) 与结合亲和力正相关

    参数:
      model: 训练好的 RTMScoreToyModel
      res_feats: (N_r, 21) tensor
      lig_feats: (N_l, 10) tensor
      dist_matrix: (N_r, N_l) tensor
      mask: (N_r, N_l) bool tensor

    返回:
      score: float -- 距离概率之和
    """
    model.eval()
    with torch.no_grad():
        pi, mu, sigma = model(res_feats, lig_feats, mask)
        y = dist_matrix[mask]
        normal = Normal(mu, sigma)
        logprob = normal.log_prob(y.unsqueeze(-1).expand_as(mu))
        logprob = logprob + torch.log(pi + 1e-10)
        prob = logprob.exp().sum(-1)   # 对高斯分量求和 -> 每对的概率
        return prob.sum().item()       # 对所有对求和 -> 单个打分值

In [ ]:
# ---- 实例化模型并查看参数量 ----

model = RTMScoreToyModel().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

total_params = sum(p.numel() for p in model.parameters())
print(f'模型参数量: {total_params:,}')
print(f'\n模型结构:')
print(model)

# 各层参数统计
layer_params = []
for name, param in model.named_parameters():
    layer_params.append({
        '层名': name,
        '形状': str(tuple(param.shape)),
        '参数量': param.numel()
    })
display(pd.DataFrame(layer_params))

## 5. 训练

训练过程的关键要点：

- **损失函数**：MDN 负对数似然（negative log-likelihood），而非 MSE
- **训练目标**：学习残基-原子距离的混合高斯分布
- **注意**：logKa 不参与训练，仅用于最终评估
- **每个 batch 只有一个样本**（`batch_size=1`），因为不同复合物的大小不同

In [ ]:
print(f'开始训练 ({N_EPOCHS} 轮)...')
print(f'训练目标: 学习残基-原子距离的混合高斯分布')
print(f'损失函数: MDN negative log-likelihood')
print('-' * 60)

for epoch in range(1, N_EPOCHS + 1):
    # ---- 训练 ----
    model.train()
    train_losses = []

    for batch in train_loader:
        res_feats, lig_feats, dist_matrix, mask, _label = batch
        # 去掉 batch 维度 (因为 BATCH_SIZE=1)
        res_feats = res_feats.squeeze(0).to(DEVICE)
        lig_feats = lig_feats.squeeze(0).to(DEVICE)
        dist_matrix = dist_matrix.squeeze(0).to(DEVICE)
        mask = mask.squeeze(0).to(DEVICE)

        # 跳过没有足够有效对的样本
        if mask.sum() < 2:
            continue

        # 前向传播
        pi, mu, sigma = model(res_feats, lig_feats, mask)

        # 训练目标是距离矩阵中的有效对
        y = dist_matrix[mask]

        # MDN 损失: 距离值在混合高斯分布下的负对数似然
        loss = mdn_loss(pi, sigma, mu, y)

        if torch.isnan(loss) or torch.isinf(loss):
            continue

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    # ---- 验证 (在测试集上计算 MDN loss) ----
    model.eval()
    val_losses = []
    with torch.no_grad():
        for batch in test_loader:
            res_feats, lig_feats, dist_matrix, mask, _label = batch
            res_feats = res_feats.squeeze(0).to(DEVICE)
            lig_feats = lig_feats.squeeze(0).to(DEVICE)
            dist_matrix = dist_matrix.squeeze(0).to(DEVICE)
            mask = mask.squeeze(0).to(DEVICE)

            if mask.sum() < 2:
                continue

            pi, mu, sigma = model(res_feats, lig_feats, mask)
            y = dist_matrix[mask]
            loss = mdn_loss(pi, sigma, mu, y)

            if not (torch.isnan(loss) or torch.isinf(loss)):
                val_losses.append(loss.item())

    # 每 20 轮打印一次训练进展
    if epoch % 20 == 0 or epoch == 1:
        avg_train = np.mean(train_losses) if train_losses else float('nan')
        avg_val = np.mean(val_losses) if val_losses else float('nan')
        print(f'  Epoch {epoch:4d}/{N_EPOCHS} | '
              f'Train MDN Loss: {avg_train:.4f} | '
              f'Val MDN Loss: {avg_val:.4f}')

print('-' * 60)
print('训练完成!')

## 6. 评估与可视化

评估流程：

1. 对每个测试复合物，用训练好的模型计算 **score**（距离概率之和）
2. 计算 score 与真实 logKa 之间的 **Pearson 相关系数**
3. 用**线性回归**将 score 映射到 logKa 的尺度，计算 **RMSE**
4. 绘制散点图

**注意**：score 本身不是亲和力预测值，而是距离似然之和。通过线性回归可以将其映射到 logKa 的尺度上。

In [ ]:
# ---- 计算测试集的 RTMScore ----

model.eval()
scores = []
true_labels = []
pdbids = []

for d in test_data:
    res_feats = torch.FloatTensor(d['res_feats']).to(DEVICE)
    lig_feats = torch.FloatTensor(d['lig_feats']).to(DEVICE)
    dist_matrix = torch.FloatTensor(d['dist_matrix']).to(DEVICE)
    mask = torch.BoolTensor(d['mask']).to(DEVICE)

    if mask.sum() < 2:
        continue

    score = compute_score(model, res_feats, lig_feats, dist_matrix, mask)
    scores.append(score)
    true_labels.append(d['label'])
    pdbids.append(d['pdbid'])

scores = np.array(scores)
true_labels = np.array(true_labels)

# 计算 Pearson 相关系数
r, p_value = pearsonr(scores, true_labels)
print(f'Pearson R: {r:.4f} (p-value: {p_value:.2e})')

# 用线性回归将 score 映射到 logKa 尺度, 计算 RMSE
reg = LinearRegression()
reg.fit(scores.reshape(-1, 1), true_labels)
pred_logKa = reg.predict(scores.reshape(-1, 1))
rmse = np.sqrt(mean_squared_error(true_labels, pred_logKa))
print(f'RMSE (线性拟合后): {rmse:.4f}')
print(f'测试样本数: {len(scores)}')

# 展示部分测试结果
result_df = pd.DataFrame({
    'PDB ID': pdbids[:10],
    'RTMScore': [f'{s:.4f}' for s in scores[:10]],
    '真实 logKa': [f'{l:.2f}' for l in true_labels[:10]],
    '预测 logKa (线性拟合)': [f'{p:.2f}' for p in pred_logKa[:10]]
})
print('\n部分测试结果:')
display(result_df)

In [ ]:
# ---- 可视化: 散点图 ----

_fig, ax = plt.subplots(1, 1, figsize=(7, 6))
ax.scatter(scores, true_labels, alpha=0.6, edgecolors='k', linewidth=0.5, s=40)

# 画线性拟合线
x_line = np.linspace(scores.min(), scores.max(), 100)
y_line = reg.predict(x_line.reshape(-1, 1))
ax.plot(x_line, y_line, 'r--', linewidth=2, label='Linear fit')

ax.set_xlabel('RTMScore (distance likelihood sum)', fontsize=12)
ax.set_ylabel('Experimental logKa', fontsize=12)
ax.set_title(f'Toy RTMScore: Pearson R = {r:.3f}, RMSE = {rmse:.3f}', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 总结

本教程演示了 RTMScore 的核心思想：

1. **训练目标**：学习残基-原子距离的混合高斯分布（MDN），而非直接回归亲和力
2. **打分方式**：将观测到的距离代入学到的分布，计算似然之和作为打分值
3. **评估方法**：通过 Pearson R 和 RMSE 评估打分与真实亲和力的相关性

### 简化版 vs 完整版 RTMScore

| 方面 | 本教程（简化版） | 完整版 RTMScore |
|------|------------------|----------------|
| 特征提取 | 简单 one-hot 编码 | 图变换器 (Graph Transformer) |
| 残基表示 | 21 维 one-hot | 深层图神经网络提取的表示 |
| 原子表示 | 10 维 one-hot | 深层图神经网络提取的表示 |
| 配对建模 | 简单拼接 + MLP | 跨图注意力机制 |
| 数据规模 | CASF-2016 (285 复合物) | PDBbind v2020 (19,443 复合物) |

### 参考文献

- Shen C, Zhang X, Deng Y, et al. Boosting Protein-Ligand Binding Pose Prediction and Virtual Screening Based on Residue-Atom Distance Likelihood Potential and Graph Transformer. *J. Med. Chem.*, 2022, 65(15): 10691-10706.